# Multicollinearity Reduction (VIF Elimination)

Step 3b of the training pipeline: removes features that mostly repeat
information other features already carry, before the models are tuned and
trained. Reads `model_dataset_initial.parquet` (from
`03_MDS-Creation/model_dataset_creation.ipynb`) and writes the reduced
`model_dataset_final.parquet` that every model notebook reads.

**What VIF is:** a feature's Variance Inflation Factor measures how well
the *other* features together can predict it. VIF = 1 means it's unique;
VIF = 10 means about 90% of it is explained by the others, i.e. it's
largely redundant. This catches more than looking at pairs of features: a
feature can be redundant with a combination of several others even when no
single one is strongly correlated with it. That's common here, since many
features are the same measure over different periods (3 years, 1 year,
6 months …).

VIF depends on which features are in the set, so it's calculated here on
the assembled dataset rather than per table in Step 2.

**Sections**
1. Settings, and each feature's points score from Step 2
2. Load the dataset and choose which features are candidates
3. How VIF is calculated
4. Remove redundant features one at a time, protecting strong ones
5. Diagnostics
6. Save
7. Free memory

**Outputs** (`data/Model-Training/`)
- `ModelDataset/model_dataset_final.parquet` — the reduced dataset (the `_initial` input is left as it was)
- `Variable-Selection/selected_features.csv` — the features kept
- `Variable-Selection/vif_elimination_log.csv` — what was dropped at each step, and why
- `Variable-Selection/correlation_matrix.xlsx`, `vif_values.xlsx` — for reference

Features listed in `feature_exceptions.csv` (written by
`model_dataset_creation.ipynb`) are treated like any other candidate here.

**Afterwards** the feature set has changed, so the models need retuning and
retraining: the `03_mds_creation_and_multicollinearity_test` Airflow DAG
does this for `LightGBM_v1` automatically; run the other versions in
`04_model_pipeline`.

## 1. Imports & Configuration

`DATA_ROOT` is the project's `data/` folder: `/data` inside Docker,
otherwise this project's folder on your PC.

- `VIF_THRESHOLD` — keep removing features until every remaining one has a VIF at or below this.
- `CORR_PARTNER_THRESHOLD` — how strongly correlated (either direction) two
  features must be to count as "partners" in step 4.
- `ALWAYS_KEEP_FEATURES` — numeric features that are never removed and are left
  out of the VIF calculation (so they can't cause others to be removed either).

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")
HPO_DIR  = os.path.join(DATA_ROOT, "Model-Training", "HyperParameters", "LightGBM_v1")   # created but not used below

os.makedirs(HPO_DIR, exist_ok=True)

PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset", "model_dataset_initial.parquet")      # input
PATH_MODEL_DATASET_OUT = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset", "model_dataset_final.parquet")   # output
VAR_SELECTION_DIR = os.path.join(DATA_ROOT, "Model-Training", "Variable-Selection")
os.makedirs(VAR_SELECTION_DIR, exist_ok=True)

PATH_SELECTED_FEATURES = os.path.join(VAR_SELECTION_DIR, "selected_features.csv")
PATH_ELIMINATION_LOG = os.path.join(VAR_SELECTION_DIR, "vif_elimination_log.csv")
PATH_CORR_MATRIX = os.path.join(VAR_SELECTION_DIR, "correlation_matrix.xlsx")
PATH_VIF_VALUES = os.path.join(VAR_SELECTION_DIR, "vif_values.xlsx")
PATH_SELECTION_METRICS = os.path.join(VAR_SELECTION_DIR, "numeric_selection_metrics_full_df.xlsx")   # from select_vars.ipynb
PATH_FEATURE_EXCEPTIONS = os.path.join(VAR_SELECTION_DIR, "feature_exceptions.csv")

# 10 is the most common rule of thumb (5 is stricter), e.g. Kutner et al.,
# "Applied Linear Statistical Models"
VIF_THRESHOLD = 10.0
CORR_PARTNER_THRESHOLD = 0.7

# Kept whatever their VIF. EXT_SCORE_MEAN (the average of EXT_SOURCE_1/2/3) has a
# VIF of ~11.6 because it overlaps the three scores it's built from; it's kept
# on purpose (added 2026-11-21) so the models can use it alongside them.
ALWAYS_KEEP_FEATURES = ['EXT_SCORE_MEAN']

### Each Feature's Points Score

Loads the `points` score (0–5) that `select_vars.ipynb` gave every feature
in Step 2 — a combined measure of Gini, Cohen's d, logistic regression
coefficient and correlation with the target. Step 4 uses it to decide
*which* feature in a redundant group to drop, so strong predictors are kept.

In [11]:
selection_metrics = pd.read_excel(PATH_SELECTION_METRICS)
feature_score = dict(zip(selection_metrics['variable'], selection_metrics['points']))

# A feature with no score (rare for numeric features) gets 0 -- the lowest
# priority, so it's the first to go if it's part of a redundant group
missing_scores = [f for f, score in feature_score.items() if score is None or np.isnan(score)]
if missing_scores:
    print(f"WARNING: {len(missing_scores)} features have no selection score "
          f"on record — defaulting to score 0 (lowest priority): {missing_scores[:5]}...")
    for f in missing_scores:
        feature_score[f] = 0

print(f"Loaded predictive scores for {len(feature_score)} features.")

Loaded predictive scores for 1227 features.


## 2. Load the Dataset

VIF only makes sense for numeric features. These are always kept and never
candidates for removal:

- text (category) columns
- the yes/no "had activity in this period" flags (columns containing `IS_IN_`)
- the numeric features in `ALWAYS_KEEP_FEATURES` (they're also left out of the
  VIF calculation, so they don't count against other features)
- `SK_ID_CURR` and `TARGET`

In [12]:
model_dataset = pd.read_parquet(PATH_MODEL_DATASET)

cat_cols = model_dataset.select_dtypes(include=['object', 'category']).columns.tolist()   # always kept
time_indicator_cols = [c for c in model_dataset.columns  if 'IS_IN_' in c]                  # always kept
exception_cols = [
    c for c in pd.read_csv(PATH_FEATURE_EXCEPTIONS)['feature']
    if c in model_dataset.columns
]
always_keep_cols = [c for c in ALWAYS_KEEP_FEATURES if c in model_dataset.columns]   # always kept
missing_keep = [c for c in ALWAYS_KEEP_FEATURES if c not in model_dataset.columns]
if missing_keep:
    print(f"WARNING: ALWAYS_KEEP_FEATURES not in the dataset: {missing_keep}")
# Everything else is a candidate for removal
numeric_cols = [
    c for c in model_dataset.columns
    if c not in cat_cols + time_indicator_cols + always_keep_cols + ['SK_ID_CURR', 'TARGET']
]

print(f"Loaded dataset       : {model_dataset.shape}")
print(f"Numeric features (candidates for elimination): {len(numeric_cols)}")
print(f"  of which exception features                 : {len(exception_cols)}")
print(f"Categorical features (always kept)           : {len(cat_cols)}")
print(f"Always-keep numeric features                 : {always_keep_cols}")

Loaded dataset       : (307511, 592)
Numeric features (candidates for elimination): 553
Categorical features (always kept)           : 10


## 3. How VIF Is Calculated

Every feature's VIF can be read straight off the correlation matrix: it's
the diagonal of the matrix's inverse.

The correlation between two features doesn't change when a third is
dropped, so the full correlation matrix (~300,000 rows × ~360 features) is
calculated **once**, with fast NumPy matrix maths. Each step of the removal
loop then just takes the rows and columns of the features still remaining
and inverts that smaller matrix, which takes milliseconds. (Recalculating
the correlations from the raw data every step took minutes per step.)

Missing values are filled with each feature's median first (step 4), since
the NumPy calculation can't handle blanks.

In [4]:
def build_full_correlation_matrix(df, feature_cols):
    """
    Correlation matrix of feature_cols, calculated once with NumPy (much
    faster than pandas' .corr()). Returned as a DataFrame so it can be
    sliced by feature name.

    df must have no missing values.
    """
    values = df[feature_cols].values.astype(np.float64)
    corr_array = np.corrcoef(values, rowvar=False)
    return pd.DataFrame(corr_array, index=feature_cols, columns=feature_cols)


def vif_from_correlation_slice(corr_df, feature_cols):
    """
    VIF of each feature in feature_cols, from the precomputed correlation matrix:
    the diagonal of the inverse of their correlation sub-matrix. (pinv copes
    with features that are exact copies of each other.)

    Returns
    -------
    pd.DataFrame with columns ['feature', 'VIF'], highest VIF first.
    """
    sub_corr = corr_df.loc[feature_cols, feature_cols]
    inverse_matrix = np.linalg.pinv(sub_corr.values)
    vifs = np.diag(inverse_matrix)
    vif_df = pd.DataFrame({'feature': feature_cols, 'VIF': vifs})
    return vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)

## 4. Remove Redundant Features, Protecting Strong Ones

Repeat until every remaining feature's VIF is at or below `VIF_THRESHOLD`:

1. Find the feature with the highest VIF.
2. Find its "partners": remaining features correlated with it at
   `CORR_PARTNER_THRESHOLD` or more (either direction).
3. Out of that feature and its partners, drop the one with the **lowest
   points score** — which may not be the flagged feature itself.

Why: VIF only says *where* there's redundancy, not which feature is more
useful. Without step 3, a strong feature (e.g. `EXT_SOURCE_1`) could be
dropped just for having the highest VIF at some point, while weaker copies
of it survive. When a weaker partner is dropped instead, that's counted as
a "protected swap".

If the flagged feature has no partners (its VIF comes from many weak
overlaps rather than one close copy), it's dropped itself.

In [5]:
from sklearn.impute import SimpleImputer

remaining_features = numeric_cols.copy()
elimination_log = []
iteration = 0
protected_swaps = 0

# Fill blanks with each feature's median (the correlation calculation can't handle blanks)
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(model_dataset[remaining_features]),
    columns=remaining_features,
)

print(f"Building correlation matrix once for {len(remaining_features)} features...")
full_corr_matrix = build_full_correlation_matrix(X_imputed, remaining_features)
print("Correlation matrix built. Starting elimination.\n")
print(f"VIF threshold: {VIF_THRESHOLD}  |  Partner correlation threshold: {CORR_PARTNER_THRESHOLD}\n")

while True:
    vif_df = vif_from_correlation_slice(full_corr_matrix, remaining_features)
    max_vif_row = vif_df.iloc[0]

    # Done once no feature is above the threshold
    if max_vif_row['VIF'] <= VIF_THRESHOLD:
        print(f"Stopped: max VIF = {max_vif_row['VIF']:.2f} "
              f"({max_vif_row['feature']}) is at or below threshold.")
        break

    flagged_feature = max_vif_row['feature']
    flagged_vif = max_vif_row['VIF']

    # Remaining features strongly correlated with the flagged one
    partner_corrs = full_corr_matrix.loc[flagged_feature, remaining_features].abs()
    partners = partner_corrs[
        (partner_corrs >= CORR_PARTNER_THRESHOLD) & (partner_corrs.index != flagged_feature)
    ].index.tolist()

    drop_pool = [flagged_feature] + partners

    # Drop the weakest of the group (lowest points score), so strong predictors are kept
    feature_to_drop = min(drop_pool, key=lambda f: feature_score.get(f, 0))

    # A "protected swap": a weaker partner was dropped instead of the flagged feature
    was_protected_swap = (feature_to_drop != flagged_feature)
    if was_protected_swap:
        protected_swaps += 1

    elimination_log.append({
        'iteration': iteration,
        'flagged_feature': flagged_feature,
        'flagged_feature_vif': flagged_vif,
        'flagged_feature_score': feature_score.get(flagged_feature, 0),
        'dropped_feature': feature_to_drop,
        'dropped_feature_score': feature_score.get(feature_to_drop, 0),
        'was_protected_swap': was_protected_swap,
        'features_remaining_after': len(remaining_features) - 1,
    })

    remaining_features.remove(feature_to_drop)
    iteration += 1

    # Progress every 10 drops
    if iteration % 10 == 0:
        print(f"  [{iteration:03d}] dropped '{feature_to_drop}' "
              f"(score={feature_score.get(feature_to_drop, 0)}) — "
              f"{len(remaining_features)} features remain")

print(f"\nElimination complete: {iteration} features dropped "
      f"({protected_swaps} were protected swaps — a weaker correlated feature "
      f"was dropped instead of the flagged one), {len(remaining_features)} features remain.")

Building correlation matrix once for 553 features...
Correlation matrix built. Starting elimination.

VIF threshold: 10.0  |  Partner correlation threshold: 0.7

  [010] dropped 'CREDITCARD_AVG_POS_DRAWING_CNT_RATIO_PREVLOAN_MEAN' (score=2) — 543 features remain
  [020] dropped 'CREDITCARD_POS_DRAWING_AMT_RATIO_MAX_MEAN_LAST3Y' (score=2) — 533 features remain
  [030] dropped 'CREDITCARD_CNT_DRAWINGS_POS_CURRENT_QOQ_ABS_CHANGE_STD_LAST1Y' (score=2) — 523 features remain
  [040] dropped 'CREDITCARD_POS_DRAWING_CNT_RATIO_MAX_LAST3M' (score=2) — 513 features remain
  [050] dropped 'CREDITCARD_AMT_RECIVABLE_MAX_LAST3M' (score=2) — 503 features remain
  [060] dropped 'CREDITCARD_AMT_BALANCE_MAX_LAST1Y' (score=2) — 493 features remain
  [070] dropped 'CREDITCARD_AMT_INST_MIN_REGULARITY_MAX_LAST6M' (score=2) — 483 features remain
  [080] dropped 'CREDITCARD_AMT_RECEIVABLE_PRINCIPAL_MEAN_LAST1Y' (score=2) — 473 features remain
  [090] dropped 'CREDITCARD_AMT_BALANCE_MEAN_LAST3M' (score=2) — 463

## 5. Diagnostics

How many features were dropped, the VIFs left, which exception features
survived, the highest-VIF drops, and examples of protected swaps.

In [15]:
final_vif = vif_from_correlation_slice(full_corr_matrix, remaining_features)

print(f"Features before elimination : {len(numeric_cols)}")
print(f"Features after elimination  : {len(remaining_features)}")
print(f"Features dropped            : {len(numeric_cols) - len(remaining_features)}")
print(f"Max VIF after elimination   : {final_vif['VIF'].max():.2f}")
print(f"Mean VIF after elimination  : {final_vif['VIF'].mean():.2f}")

surviving_exceptions = [f for f in remaining_features if f in exception_cols]
print(f"Exception features kept     : {len(surviving_exceptions)} of {len(exception_cols)}")
for f in surviving_exceptions:
    print(f"    {f}")

elimination_log_df = pd.DataFrame(elimination_log)

if len(elimination_log_df) > 0:
    # The 15 steps with the highest flagged VIF
    print("\nTop 15 highest-VIF drops (by the flagged feature's VIF at that step):")
    print(
        elimination_log_df
        .sort_values('flagged_feature_vif', ascending=False)
        .head(15)[['flagged_feature', 'flagged_feature_vif', 'dropped_feature']]
        .to_string(index=False, float_format='{:.1f}'.format)
    )

protected_examples = (
    elimination_log_df[elimination_log_df['was_protected_swap']]
    if len(elimination_log_df) > 0 else pd.DataFrame()
)
if len(protected_examples) > 0:
    print(f"\n{len(protected_examples)} protected swaps occurred — "
          f"strong features that were saved by dropping a weaker correlated partner instead:")
    print(
        protected_examples[
            ['flagged_feature', 'flagged_feature_score', 'dropped_feature', 'dropped_feature_score']
        ]
        .head(10)
        .to_string(index=False)
    )
else:
    print("\nNo protected swaps occurred — every dropped feature was the flagged "
          "highest-VIF feature with no stronger correlated partner to protect.")

Features before elimination : 179
Features after elimination  : 179
Features dropped            : 0
Max VIF after elimination   : 9.50
Mean VIF after elimination  : 3.76

No protected swaps occurred — every dropped feature was the flagged highest-VIF feature with no stronger correlated partner to protect.


Idea, not in use: bring back dropped features that scored 4 or more.

In [16]:
#additional_features_from_protected = protected_examples[protected_examples['dropped_feature_score']>=4].reset_index()

## 6. Save

- `model_dataset_final.parquet` — `SK_ID_CURR`, the kept numeric features,
  the `ALWAYS_KEEP_FEATURES`, the always-kept text columns and `IS_IN_` flags, and `TARGET`. Every
  model notebook reads this file.
- `selected_features.csv` — the kept feature names.
- `vif_elimination_log.csv` — each step: the flagged feature and its VIF,
  what was dropped, and both scores. Useful if you need to explain why a
  feature isn't in the models.
- `correlation_matrix.xlsx` and `vif_values.xlsx` (the remaining features' final VIFs).

In [ ]:
final_feature_cols = remaining_features + always_keep_cols + cat_cols + time_indicator_cols
reduced_dataset = model_dataset[['SK_ID_CURR'] + final_feature_cols + ['TARGET']]

full_corr_matrix.to_excel(PATH_CORR_MATRIX, index=True)
print(f"Saved correlation matrix → {PATH_CORR_MATRIX}")

final_vif.to_excel(PATH_VIF_VALUES, index=False)
print(f"Saved final VIF values → {PATH_VIF_VALUES}")

reduced_dataset.to_parquet(PATH_MODEL_DATASET_OUT, index=False)
print(f"Saved reduced model_dataset.parquet ({reduced_dataset.shape}) → {PATH_MODEL_DATASET_OUT}")

pd.Series(final_feature_cols).to_csv(PATH_SELECTED_FEATURES, index=False)
print(f"Saved updated feature list → {PATH_SELECTED_FEATURES}")

elimination_log_df.to_csv(PATH_ELIMINATION_LOG, index=False)
print(f"Saved elimination log ({len(elimination_log_df)} rows) → {PATH_ELIMINATION_LOG}")

Saved reduced model_dataset.parquet ((307511, 218)) → a:\Dominic\My Projects\Home Credit\data\Model-Training\ModelDataset\model_dataset.parquet
Saved updated feature list → a:\Dominic\My Projects\Home Credit\data\Model-Training\Variable-Selection\selected_features.csv
Saved elimination log (374 rows) → a:\Dominic\My Projects\Home Credit\data\Model-Training\Variable-Selection\vif_elimination_log.csv


## 7. Free Memory

In [9]:
del model_dataset, X_imputed, reduced_dataset
import gc; gc.collect()
print("All DataFrames released.")

print("\nNext step: re-run 02_hyperparameter_tuning.ipynb and "
      "03_training_and_analysis.ipynb against the reduced feature set.")

All DataFrames released.

Next step: re-run 02_hyperparameter_tuning.ipynb and 03_training_and_analysis.ipynb against the reduced feature set.
